# Lábia — treino SFT do aluno (`sft`)

Este notebook treina o **Qwen3-4B-Instruct-2507** com LoRA. Variante: o modelo principal: treinado só com as conversas do professor que passaram no **filtro duplo** do verificador.

Etapas:

1. junta as duas fatias da geração (anexadas como fonte deste notebook);
2. mede a velocidade das configurações de treino aqui mesmo e treina com a mais rápida;
3. **funde** o adaptador LoRA nos pesos do modelo: o resultado é um modelo único, que não depende mais do adaptador;
4. converte para **GGUF de 8 bits**, o formato que a avaliação (Ollama) e a demo (CPU) usam.

O treino para sozinho ao bater no orçamento de 8 horas e salva o que aprendeu até ali.

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path
LABIA_COMMIT = "ab3c24ec6dbc0739b4f414b5dfb08aa5c3fea9c7"
VARIANT = "sft"
DATA_FILE = "sft.jsonl"
TRAIN_CMD = ["python", "-m", "labia.train_sft"]
EPOCHS = "2"
BUDGET_H = "8"
BASE = "Qwen/Qwen3-4B-Instruct-2507"
WORK = Path("/kaggle/working")
ADAPTER = WORK / f"adaptador-{VARIANT}"
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes",
                    f"git+https://github.com/deniscarvalho-tech/labia.git@{LABIA_COMMIT}"], capture_output=True, text=True)
print((r.stdout + r.stderr)[-400:])
if "--unsloth" in TRAIN_CMD:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth"], capture_output=True, text=True)
    print((r.stdout + r.stderr)[-400:])

## 1. Juntar as fatias da geração

In [ ]:
parts = sorted(Path("/kaggle/input").rglob(DATA_FILE))
print([str(p) for p in parts])
assert len(parts) == 2, "esperava as duas fatias da geração anexadas como fonte"
rows = [line for p in parts for line in p.read_text(encoding="utf-8").splitlines() if line.strip()]
data = WORK / "treino.jsonl"
data.write_text("\n".join(rows) + "\n", encoding="utf-8")
from collections import Counter
meta = [json.loads(r)["meta"] for r in rows]
print(len(rows), "conversas;", dict(Counter(m["tactic"] for m in meta)), dict(Counter("legitimo" if m["legit"] else "golpe" for m in meta)))

## 2. Escolher a configuração mais rápida, medindo aqui mesmo

Rodamos 3 passos de cada configuração com as **16 conversas mais longas** dos dados reais, que são o pior caso de memória. Fica a mais rápida que não falhou. O Unsloth só é testado se nenhuma configuração comum passar de 400 tokens por segundo, porque a instalação dele troca versões de bibliotecas.

Depois, o número de passadas pelos dados (as **épocas**) se ajusta à velocidade medida: no máximo 2, ou o que couber em 85% do orçamento de tempo. Assim o ciclo da taxa de aprendizado termina inteiro, em vez de o treino ser cortado no meio.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], capture_output=True)  # a 0.10 do Kaggle quebra o LoRA da peft
ONE = {"CUDA_VISIBLE_DEVICES": "0"}
PROBES = {
    "lora16-2gpus": (["torchrun", "--nproc_per_node", "2", "-m", "labia.train_sft", "--mode", "lora16"], {}),
    "qlora-2gpus": (["torchrun", "--nproc_per_node", "2", "-m", "labia.train_sft", "--mode", "qlora"], {}),
    "lora16-1gpu": ([sys.executable, "-m", "labia.train_sft", "--mode", "lora16"], ONE),
}
UNSLOTH = ("unsloth-qlora-1gpu", [sys.executable, "-m", "labia.train_sft", "--unsloth", "--mode", "qlora"], ONE)
speed = {}


def probe(name, cmd, env_extra):
    out = WORK / "sondas" / name
    p = subprocess.run(cmd + ["--data", str(data), "--out", str(out), "--max-steps", "3", "--grad-accum", "4",
                              "--limit", "16", "--longest"], capture_output=True, text=True, env=dict(os.environ, **env_extra))
    stats = out / "treino.json"
    if p.returncode != 0 or not stats.exists():
        errors = [l for l in (p.stdout + p.stderr).splitlines() if "Error" in l or "error" in l][:6]
        print(f"[{name}] falhou: " + " | ".join(errors)[-1200:])
        return
    s = json.loads(stats.read_text())
    speed[name] = (s["tokens_por_segundo"], cmd, env_extra)
    print(f"[{name}] {s['tokens_por_segundo']} tokens/s, pico {s['pico_memoria_gb']} GB")


for name, (cmd, env_extra) in PROBES.items():
    probe(name, cmd, env_extra)
if not speed or max(v[0] for v in speed.values()) < 400:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth"], capture_output=True, text=True)
    print((r.stdout + r.stderr)[-300:])
    probe(*UNSLOTH)
assert speed, "nenhuma configuração de treino funcionou"
best = max(speed, key=lambda k: speed[k][0])
tokens_per_s, TRAIN_CMD, TRAIN_ENV = speed[best]
from labia.train_sft import encode_all, load_examples
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(BASE)
encoded, data_stats = encode_all(tok, load_examples(data), 6144)
seconds_per_epoch = data_stats["tokens_total"] / tokens_per_s
EPOCHS = str(round(min(2.0, 0.85 * float(BUDGET_H) * 3600 / seconds_per_epoch), 2))
print(f"escolhida: {best} ({tokens_per_s} tokens/s); {data_stats['tokens_total']:,} tokens por época "
      f"= {seconds_per_epoch / 3600:.1f} h; épocas: {EPOCHS}")

## 3. Treinar

In [ ]:
start = time.perf_counter()
cmd = TRAIN_CMD + ["--data", str(data), "--out", str(ADAPTER), "--epochs", EPOCHS, "--time-budget-h", BUDGET_H]
print(" ".join(cmd))
p = subprocess.run(cmd, capture_output=True, text=True, env=dict(os.environ, **TRAIN_ENV))
print(p.stdout[-3000:])
if p.returncode != 0:
    print(p.stderr[-4000:])
    raise SystemExit("o treino falhou")
print(f"treino em {(time.perf_counter() - start) / 3600:.2f} h")
print(json.dumps({k: v for k, v in json.loads((ADAPTER / "treino.json").read_text()).items() if k != "log"}, indent=1))

## 4. Fundir o adaptador e converter para GGUF de 8 bits

A fusão roda na CPU (o modelo em float16 ocupa 8 GB de RAM; a máquina tem 31 GB) para não disputar a GPU. A conversão usa o script oficial do llama.cpp.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

merged = Path("/tmp/merged")
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map={"": "cpu"})
model = PeftModel.from_pretrained(model, str(ADAPTER)).merge_and_unload()
model.save_pretrained(merged, safe_serialization=True)
AutoTokenizer.from_pretrained(BASE).save_pretrained(merged)
del model
subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/ggml-org/llama.cpp", "/tmp/llama.cpp"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "/tmp/llama.cpp/gguf-py", "sentencepiece"], check=True)
gguf = WORK / f"labia-{VARIANT}-q8_0.gguf"
subprocess.run([sys.executable, "/tmp/llama.cpp/convert_hf_to_gguf.py", str(merged), "--outtype", "q8_0", "--outfile", str(gguf)],
               check=True, stdout=subprocess.DEVNULL)
print(gguf.name, f"{gguf.stat().st_size / 1024**3:.2f} GB")
for f in sorted(WORK.iterdir()):
    print(f.name)